<a href="https://colab.research.google.com/github/Danial-071/langchain-project/blob/main/LangChain%20Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [150]:
# !pip install langchain_classic
# !pip install langchain_core
# !pip install langchain-community
# !pip install langchain-openrouter
!pip install langgraph

In [161]:
from langchain_classic.schema.runnable import RunnableParallel
from langchain_classic.prompts import PromptTemplate
from langchain_openrouter import ChatOpenRouter
from langchain_core.tools import tool
from langchain_classic.agents import create_openai_tools_agent
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import seaborn
import pandas as pd
import glob
import os
from typing import List, Dict, Any


import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get("API")

model = ChatOpenRouter(model = userdata.get("model"))

template = PromptTemplate.from_template("Hello {name}!")


In [136]:
from typing import Optional, List

@tool
def list_csv_files() -> Optional[List[str]]:
    """List all CSV file names in the local directory.

    Returns:
        A list containing CSV file names.
        If no CSV files are found, returns None.
    """
    csv_files = glob.glob(os.path.join(os.getcwd(), "*.csv"))
    if not csv_files:
        return None
    return [os.path.basename(file) for file in csv_files]

In [137]:
print("Tool Name:", list_csv_files.name)
print("Tool Description:", list_csv_files.description)
print("Tool Arguments:", list_csv_files.args)

Tool Name: list_csv_files
Tool Description: List all CSV file names in the local directory.

Returns:
    A list containing CSV file names.
    If no CSV files are found, returns None.
Tool Arguments: {}


In [138]:
DATAFRAME_CACHE = {}

@tool
def preload_datasets(paths: List[str]) -> str:
    """
    Loads CSV files into a global cache if not already loaded.

    This function helps to efficiently manage datasets by loading them once
    and storing them in memory for future use. Without caching, you would
    waste tokens describing dataset contents repeatedly in agent responses.

    Args:
        paths: A list of file paths to CSV files.

    Returns:
        A message summarizing which datasets were loaded or already cached.
    """
    loaded = []
    cached = []
    for path in paths:
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)
            loaded.append(path)
        else:
            cached.append(path)

    return (
        f"Loaded datasets: {loaded}\n"
        f"Already cached: {cached}"
    )

In [139]:
from typing import List, Optional,Dict,Any

@tool
def get_dataset_summaries(dataset_paths: List[str]) -> List[Dict[str, Any]]:
    """
    Analyze multiple CSV files and return metadata summaries for each.

    Args:
        dataset_paths (List[str]):
            A list of file paths to CSV datasets.

    Returns:
        List[Dict[str, Any]]:
            A list of summaries, one per dataset, each containing:
            - "file_name": The path of the dataset file.
            - "column_names": A list of column names in the dataset.
            - "data_types": A dictionary mapping column names to their data types (as strings).
    """
    summaries = []

    for path in dataset_paths:
        # Load and cache the dataset if not already cached
        if path not in DATAFRAME_CACHE:
            DATAFRAME_CACHE[path] = pd.read_csv(path)

        df = DATAFRAME_CACHE[path]

        # Build summary
        summary = {
            "file_name": path,
            "column_names": df.columns.tolist(),
            "data_types": df.dtypes.astype(str).to_dict()
        }

        summaries.append(summary)

    return summaries

In [140]:
@tool
def call_dataframe_method(file_name: str, method: str) -> str:
   """
   Execute a method on a DataFrame and return the result.
   This tool lets you run simple DataFrame methods like 'head', 'tail', or 'describe'
   on a dataset that has already been loaded and cached using 'preload_datasets'.
   Args:
       file_name (str): The path or name of the dataset in the global cache.
       method (str): The name of the method to call on the DataFrame. Only no-argument
                     methods are supported (e.g., 'head', 'describe', 'info').
   Returns:
       str: The output of the method as a formatted string, or an error message if
            the dataset is not found or the method is invalid.
   Example:
       call_dataframe_method(file_name="data.csv", method="head")
   """
   # Try to get the DataFrame from cache, or load it if not already cached
   if file_name not in DATAFRAME_CACHE:
       try:
           DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
       except FileNotFoundError:
           return f"DataFrame '{file_name}' not found in cache or on disk."
       except Exception as e:
           return f"Error loading '{file_name}': {str(e)}"

   df = DATAFRAME_CACHE[file_name]
   func = getattr(df, method, None)
   if not callable(func):
       return f"'{method}' is not a valid method of DataFrame."
   try:
       result = func()
       return str(result)
   except Exception as e:
       return f"Error calling '{method}' on '{file_name}': {str(e)}"

In [141]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error

# Assumes this global cache is shared
DATAFRAME_CACHE = {}

@tool
def evaluate_classification_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a classifier on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the classification target.
    Returns:
        Dict[str, float]: A dictionary with the model's accuracy score.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}

    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}

    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return {"accuracy": acc}

@tool
def evaluate_regression_dataset(file_name: str, target_column: str) -> Dict[str, float]:
    """
    Train and evaluate a regression model on a dataset using the specified target column.
    Args:
        file_name (str): The name or path of the dataset stored in DATAFRAME_CACHE.
        target_column (str): The name of the column to use as the regression target.
    Returns:
        Dict[str, float]: A dictionary with R² score and Mean Squared Error.
    """
    # Try to get the DataFrame from cache, or load it if not already cached
    if file_name not in DATAFRAME_CACHE:
        try:
            DATAFRAME_CACHE[file_name] = pd.read_csv(file_name)
        except FileNotFoundError:
            return {"error": f"DataFrame '{file_name}' not found in cache or on disk."}
        except Exception as e:
            return {"error": f"Error loading '{file_name}': {str(e)}"}

    df = DATAFRAME_CACHE[file_name]
    if target_column not in df.columns:
        return {"error": f"Target column '{target_column}' not found in '{file_name}'."}

    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    return {
        "r2_score": r2,
        "mean_squared_error": mse
    }

In [145]:
from langchain_core.prompts import ChatPromptTemplate

# 🧠 Step 2: Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data science assistant. Use the available tools to analyze CSV files. "
     "Your job is to determine whether each dataset is for classification or regression, based on its structure."),

    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")  # Required for tool-calling agents
])

In [164]:
from langchain_core.output_parsers import StrOutputParser

tools=[list_csv_files, preload_datasets, get_dataset_summaries, call_dataframe_method, evaluate_classification_dataset, evaluate_regression_dataset]
agent = create_openai_tools_agent(llm = model, tools = tools , prompt = prompt)

# Invoke the raw tool-calling agent once.
# This returns the agent's next action (or a final result); it does not run the full tool loop.
agent_response = agent.invoke({
     "input": "What columns are in the dataset?",
    "intermediate_steps": []
})

print("Raw agent response:")
print(agent_response)


Raw agent response:
[ToolAgentAction(tool='list_csv_files', tool_input={}, log='\nInvoking: `list_csv_files` with `{}`\n\n\n', message_log=[AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking about columns in "the dataset" but hasn\'t specified which dataset. I need to first check what CSV files are available in the directory. Let me list the CSV files first.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking about columns in "the dataset" but hasn\'t specified which dataset. I need to first check what CSV files are available in the directory. Let me list the CSV files first.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1794388443-lYFYQWk4ouFoZOhoFWTI', 'created': 1794388443, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost':

In [167]:
lcel_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data science assistant. Answer questions clearly and concisely."),
    ("user", "{input}")
])

# Build and run the LCEL chain: prompt -> language model -> string parser.
chain = lcel_prompt | model | StrOutputParser()

print("Running LCEL chain:")
lcel_response = analysis_chain.invoke({
    "input": "What columns are in the dataset?"
})
print(lcel_response[:400])


Running LCEL chain:
I don't have access to a specific dataset in our conversation. Could you please clarify which dataset you're referring to? For example:
- A file you've uploaded
- A well-known dataset (like Titanic, Iris, etc.)
- A database table you're working with

Once you specify the dataset, I can help you identify its columns.


In [168]:
# Inspect the raw agent result, not the string returned by StrOutputParser.
# The raw agent returns either a list of tool actions or a final AgentFinish result.
if isinstance(agent_response, list) and agent_response:
    action = agent_response[0]

    if hasattr(action, "tool"):
        print("Agent decided to call a tool:")
        print("Tool Name:", action.tool)
        print("Tool Input:", action.tool_input)
        print("Log:\\n", action.log.strip())
    else:
        print("Agent returned:", action)

elif hasattr(agent_response, "return_values"):
    print("Agent returned a final response:")
    print(agent_response.return_values)

else:
    print("Unexpected agent response type:", type(agent_response).__name__)
    print(agent_response)


Agent decided to call a tool:
Tool Name: list_csv_files
Tool Input: {}
Log:\n Invoking: `list_csv_files` with `{}`
